# LLM-as-Judge Evaluation with Custom Models

This notebook demonstrates how to evaluate custom and fine-tuned models using `LLMAsJudgeEvaluator`.

The SDK transparently handles custom models by routing through an InspectAI-based inference path.
You use the same familiar API — the SDK detects your model type and configures everything automatically.

**What this notebook covers:**
- **Section A**: Evaluate a fine-tuned model via Model Package ARN
- **Section B**: Evaluate a Nova model (auto-routed to InspectAI+Bedrock)
- **Section C**: Monitor and retrieve existing evaluations


## Configuration

Set your AWS region, S3 bucket, and model identifiers below.

In [ ]:
# AWS Configuration
REGION = "us-east-1"
S3_BUCKET = "s3://<your-bucket>/llmaj-custom-model-eval/"

# Model Package ARN for your fine-tuned model (Section A)
MODEL_PACKAGE_ARN = "arn:aws:sagemaker:us-east-1:<account-id>:model-package/<package-name>/<version>"

# Nova model for auto-routed Bedrock evaluation (Section B)
NOVA_MODEL = "nova-textgeneration-lite"

# Judge model (evaluator) — used in both sections
EVALUATOR_MODEL = "amazon.nova-pro-v1:0"

# Dataset — S3 URI to a JSONL file with "prompt" or "query" field per line
DATASET = "s3://<your-bucket>/datasets/eval_prompts.jsonl"

# Optional: MLflow tracking server ARN
MLFLOW_ARN = "arn:aws:sagemaker:us-east-1:<account-id>:mlflow-tracking-server/<server-name>"

## Imports and Setup

In [ ]:
import json
from sagemaker.train.evaluate import LLMAsJudgeEvaluator, EvaluationPipelineExecution
from rich.pretty import pprint

# Configure logging to show INFO messages
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s'
)

---

## Section A: Evaluate a Fine-Tuned Model via Model Package ARN

When you pass a Model Package ARN as the `model` parameter, the SDK automatically:
1. Detects it as a custom model
2. Resolves model artifacts (model data URI and inference image) from the model package
3. Routes through the InspectAI inference path — deploying a temporary endpoint, running inference, then cleaning up
4. Passes the inference output to the LLM-as-Judge Phase 2 for scoring

No additional configuration needed — just use the same API as with JumpStart models.

In [ ]:
# Create evaluator with a fine-tuned model package
evaluator_finetuned = LLMAsJudgeEvaluator(
    model=MODEL_PACKAGE_ARN,
    evaluator_model=EVALUATOR_MODEL,
    dataset=DATASET,
    builtin_metrics=["Correctness", "Helpfulness", "Faithfulness"],
    mlflow_resource_arn=MLFLOW_ARN,
    s3_output_path=S3_BUCKET,
)

pprint(evaluator_finetuned)

In [ ]:
# Start the evaluation
execution_a = evaluator_finetuned.evaluate()

print(f"Evaluation started!")
print(f"  ARN: {execution_a.arn}")
print(f"  Name: {execution_a.name}")
print(f"  Status: {execution_a.status.overall_status}")

In [ ]:
# Wait for completion (polls every 30 seconds, timeout after 2 hours)
execution_a.wait(poll=30, timeout=7200)

print(f"Final status: {execution_a.status.overall_status}")

In [ ]:
# View evaluation results
execution_a.show_results(limit=10, offset=0, show_explanations=False)

---

## Section B: Evaluate a Nova Model (Auto-Routed to Bedrock)

Nova JumpStart models are automatically routed through the InspectAI+Bedrock inference path.
No special configuration is needed — just pass the Nova model name as `model`.

The SDK automatically:
1. Detects the model is a Nova JumpStart model
2. Derives the correct Bedrock cross-region inference profile ID from your session region
3. Routes through InspectAI to call Bedrock for inference
4. Passes responses to the LLM-as-Judge Phase 2 for scoring

The same API works for all model types — the routing is completely transparent.

In [ ]:
# Create evaluator with a Nova model — auto-routes to InspectAI+Bedrock
evaluator_nova = LLMAsJudgeEvaluator(
    model=NOVA_MODEL,
    evaluator_model=EVALUATOR_MODEL,
    dataset=DATASET,
    builtin_metrics=["Correctness", "Helpfulness"],
    mlflow_resource_arn=MLFLOW_ARN,
    s3_output_path=S3_BUCKET,
)

pprint(evaluator_nova)

In [ ]:
# Start the evaluation
execution_b = evaluator_nova.evaluate()

print(f"Evaluation started!")
print(f"  ARN: {execution_b.arn}")
print(f"  Name: {execution_b.name}")
print(f"  Status: {execution_b.status.overall_status}")

In [ ]:
# Wait for completion
execution_b.wait(poll=30, timeout=7200)

print(f"Final status: {execution_b.status.overall_status}")

In [ ]:
# View evaluation results
execution_b.show_results(limit=10, offset=0, show_explanations=False)

---

## Section C: Monitor and Retrieve Existing Evaluations

You can retrieve any existing evaluation by its ARN, or list all LLM-as-Judge evaluations.

### Retrieve a specific evaluation by ARN

In [ ]:
# Replace with your actual pipeline execution ARN
existing_arn = "arn:aws:sagemaker:us-east-1:<account-id>:pipeline/<pipeline-name>/execution/<execution-id>"

existing_execution = EvaluationPipelineExecution.get(
    arn=existing_arn,
    region=REGION
)

pprint(existing_execution.status)
existing_execution.show_results(limit=5, offset=0, show_explanations=False)

### List all LLM-as-Judge evaluations

In [ ]:
# Get all LLM-as-Judge evaluations
all_executions = list(LLMAsJudgeEvaluator.get_all(region=REGION))

print(f"Found {len(all_executions)} LLM-as-Judge evaluation jobs")
for exec_item in all_executions:
    print(f"  - {exec_item.name}: {exec_item.status.overall_status}")

### Refresh status of a running evaluation

In [ ]:
# Refresh and display step-level status
execution_b.refresh()
pprint(execution_b.status)

---

## Cost Implications

When using custom models or Nova models, the evaluation runs through the InspectAI inference path.
This incurs the following costs:

| Cost Component | Description |
|----------------|-------------|
| **InspectAI Orchestrator** | A SageMaker Training instance (`ml.m5.large`, ~$0.12/hr) runs the InspectAI container that orchestrates inference. This instance runs for the duration of inference generation. |
| **Inference Costs** | Depends on your model type: |
| — Nova model (auto-routed) | Standard Bedrock per-token pricing for the Nova model. The SDK derives the correct Bedrock inference profile from your region. |
| — Fine-tuned model (Model Package) | A temporary SageMaker endpoint is created for inference and automatically cleaned up after completion. You are charged for the endpoint instance time. |
| **LLM-as-Judge (Phase 2)** | Standard Bedrock pricing for the judge model (`evaluator_model`) to score responses. This cost is the same as the standard LLMAJ path. |

**Tips to manage costs:**
- Use a small dataset (5-20 samples) for initial testing
- Choose cost-effective models: `nova-textgeneration-lite` for inference, `amazon.nova-pro-v1:0` for judging
- The InspectAI orchestrator instance is minimal cost compared to inference and judging
- Endpoints are automatically cleaned up — no manual intervention needed

---

## Dataset Format

Your evaluation dataset should be a JSONL file where each line contains a `"prompt"` or `"query"` field:

```json
{"prompt": "What is machine learning?"}
{"prompt": "Explain the difference between supervised and unsupervised learning."}
{"prompt": "How does a neural network work?"}
```

The SDK automatically converts this to the InspectAI format internally. No manual reformatting is needed.